In [1]:
import pandas as pd
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

In [2]:
df = pd.read_csv(r'D:\Python\我的AI作品集\專案1_數學函數辨識\data\v2\function_dataset_cleaned_v2.csv')

In [3]:
# 定義特徵與目標變數
X = df.drop(columns=['label'])
y = df['label']

## 將數據分割為訓練集與測試集

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10, stratify=y)

In [5]:
print(f"訓練集類別:\n{y_train.value_counts()}\n")

訓練集類別:
label
cubic          400
cosine         400
exponential    400
quadratic      400
reciprocal     400
sine           400
logarithmic    400
linear         400
Name: count, dtype: int64



In [6]:
print(f"測試集類別:\n{y_test.value_counts()}\n")

測試集類別:
label
quadratic      100
logarithmic    100
reciprocal     100
sine           100
cosine         100
exponential    100
cubic          100
linear         100
Name: count, dtype: int64



## 蒐集模型, 建立pipeline

In [7]:
models = {
    "LogisticRegression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    ),
    "DecisionTree": DecisionTreeClassifier(random_state=10),

    "RandomForest": RandomForestClassifier(random_state=10),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=3)
    ),
    "SVM": make_pipeline(
        StandardScaler(),
        SVC()
    )
}

In [8]:
for name, model in models.items():
    start_time = time.perf_counter()        # 開始計時
    model.fit(X_train, y_train)
    end_time = time.perf_counter()          # 結束計時
    elapsed_time = end_time - start_time    # 計算經過時間
    print(f"{name} 模型訓練完成, 耗時: {elapsed_time:.3f}秒\n")

    # 儲存到joblib
    joblib.dump(model, rf'D:\Python\我的AI作品集\專案1_數學函數辨識\models\v2\{name}_baseline_v2.joblib')

LogisticRegression 模型訓練完成, 耗時: 0.044秒

DecisionTree 模型訓練完成, 耗時: 0.047秒

RandomForest 模型訓練完成, 耗時: 1.015秒

KNN 模型訓練完成, 耗時: 0.007秒

SVM 模型訓練完成, 耗時: 0.209秒



## 預測模型

In [9]:
from sklearn.metrics import *

In [10]:
results = []
trained_models = {}

In [11]:
for name, model in models.items():
    model = joblib.load(rf'D:\Python\我的AI作品集\專案1_數學函數辨識\models\v2\{name}_baseline_v2.joblib')
    trained_models[name] = model        # 儲存模型物件

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"{name} 模型計算準確度: {acc:.4f}")
    print(f"{name} 模型的混淆矩陣:\n{confusion_matrix(y_test, y_pred)}\n")
    print(f"{name} 模型的分類報告:\n{classification_report(y_test, y_pred)}\n")
    results.append({'Model': name,
                    'Accuracy': acc,
                    'Precision': pre,
                    'Recall': recall,
                    'F1': f1})

LogisticRegression 模型計算準確度: 0.2162
LogisticRegression 模型的混淆矩陣:
[[43 18  1  7  3  7  0 21]
 [12 25  5 25  7 16  0 10]
 [ 9 19  7 27 26  6  5  1]
 [13 19  1 26 29  7  5  0]
 [ 0  3  0 47 46  3  1  0]
 [12 31  4 27  2 15  2  7]
 [ 7 24  1 21 28 13  2  4]
 [27 19  2 23  8  9  3  9]]

LogisticRegression 模型的分類報告:
              precision    recall  f1-score   support

      cosine       0.35      0.43      0.39       100
       cubic       0.16      0.25      0.19       100
 exponential       0.33      0.07      0.12       100
      linear       0.13      0.26      0.17       100
 logarithmic       0.31      0.46      0.37       100
   quadratic       0.20      0.15      0.17       100
  reciprocal       0.11      0.02      0.03       100
        sine       0.17      0.09      0.12       100

    accuracy                           0.22       800
   macro avg       0.22      0.22      0.19       800
weighted avg       0.22      0.22      0.19       800


DecisionTree 模型計算準確度: 0.6025
DecisionTr

In [12]:
results_df = pd.DataFrame(results)

In [13]:
# 儲存成網頁
results_df.to_html(r'D:\Python\我的AI作品集\專案1_數學函數辨識\data\v2\Model_Comparison_v2.html')

## 模型比較

In [ ]:
print("模型效能比較:")
print(results_df.to_string(index=False, formatters={'Accuracy': '{:.4f}'.format,
                                                    'Precision': '{:.4f}'.format,
                                                    'Recall': '{:.4f}'.format,
                                                    'F1': '{:.4yf}'.format}))

模型效能比較:
             Model Accuracy Precision Recall     F1
LogisticRegression   0.2162    0.2199 0.2162 0.1949
      DecisionTree   0.6025    0.6065 0.6025 0.6026
      RandomForest   0.7425    0.7466 0.7425 0.7416
               KNN   0.7137    0.7250 0.7137 0.7165
               SVM   0.5900    0.6912 0.5900 0.6014


### 找出 F1 最高的模型

In [15]:
# 找出 F1 最高的模型
best_result = results_df.loc[results_df['F1'].idxmax()]
best_model_name = best_result['Model']
best_model = trained_models[best_model_name]
print(f"\n最佳模型:\n{best_result}")


最佳模型:
Model        RandomForest
Accuracy           0.7425
Precision        0.746631
Recall             0.7425
F1               0.741603
Name: 2, dtype: object


In [16]:
joblib.dump(best_model, r'D:\Python\我的AI作品集\專案1_數學函數辨識\models\v2\best_model_v2.joblib')

['D:\\Python\\我的AI作品集\\專案1_數學函數辨識\\models\\v2\\best_model_v2.joblib']